In [ ]:
import numpy as np
import pandas as pd
import streamlit as st
import pickle
import spotipy

from spotipy.oauth2 import SpotifyClientCredentials

client_id = '4c9f335caf3e4ff196795da079c44c1e'
client_secret = '7efa687e7f9049e486ec839860b409db'
client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)



def display_search_results(search_query):
    results = sp.search(q=search_query, limit=10, type='track')
    track_list = results['tracks']['items']

    for i, track in enumerate(track_list):
        st.write(i+1, '. ', track['name'], 'by', track['artists'][0]['name'])

    track_ids = [track['id'] for track in track_list]
    return track_ids

if 'current_step' not in st.session_state:
    st.session_state.current_step = 1

if 'search_query' not in st.session_state:
    st.session_state.search_query = ''

if 'single_track_id' not in st.session_state:
    st.session_state.single_track_id = ''

def main():
    st.title("HIT OR MISS?")

    if st.session_state.current_step == 1:
        search_query = st.text_input("Enter a song name:")
        if search_query:
            st.session_state.search_query = search_query
        if st.button('Display results'):
            st.session_state.current_step = 2

    elif st.session_state.current_step == 2:
        search_query = st.session_state.search_query
        st.write('Search results for',search_query, ':')
        track_ids = display_search_results(search_query)
        st.session_state.track_ids = track_ids
        #st.write(track_ids)

        track_number = st.selectbox("Select a song number:", range(1, len(track_ids) + 1))
        #st.write(track_number)

        if st.button('Predict if this song is a hit!'):
            st.session_state.single_track_id = st.session_state.track_ids[track_number-1]
            st.session_state.current_step = 3

    elif st.session_state.current_step == 3:
        st.write('This song has all these features:')
        single_track_id = st.session_state.single_track_id
        audio_features = sp.audio_features(single_track_id)

        df_test = pd.DataFrame(audio_features)
        df_test['linear_loudness'] = np.power(10, df_test['loudness'] / 10)

        set_A = ['danceability','energy','linear_loudness','speechiness','acousticness','instrumentalness','liveness','tempo','valence','duration_ms']
        df_test = df_test[set_A]

        st.write(df_test)

        song_array = df_test.iloc[0].to_numpy()
        song_reshaped = song_array.reshape(1,-1)

        pickled_model = pickle.load(open('model.pkl', 'rb'))
        pred = pickled_model.predict(song_reshaped)

        #st.write(pred)
        if pred == 0:
            st.write("It's a Dud...")
        else:
            st.write("It's a Hit!")

        if st.button('Try another song!'):
            st.session_state.current_step = 1

if __name__ == "__main__":
    main()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.1/250.1 kB 8.8 MB/s eta 0:00:00


2023-08-22 14:37:04.308 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2023-08-22 14:37:04.360 
  command:

    streamlit run /usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py [ARGUMENTS]


AttributeError: ignored